# Transformer 从零实现

这份 notebook 用 PyTorch 从零写一个完整的 **Encoder-Decoder Transformer**。

主线很简单：

1. token id 变成向量：`Embedding`
2. 加入顺序信息：`PositionalEncoding`
3. 用注意力让 token 之间交换信息：`MultiHeadAttention`
4. 用前馈网络加工每个位置：`FeedForward`
5. 堆叠出 Encoder 和 Decoder
6. 用 mask 屏蔽 padding 和未来 token
7. 跑通前向传播、loss、一个小训练循环和贪心解码

> 当前版本：2026-07-04 修正版，已修复 greedy decoding 的 print 语法，并加入模块测试。


## 1. 导入依赖

这里只用 PyTorch 和 Python 标准库。为了方便复现，先固定随机种子，并自动选择 CPU 或 GPU。


In [1]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cpu'

## 2. 配置参数

把模型尺寸集中放进配置类，后面每个模块只从 `cfg` 里取参数。

- `d_model`：每个 token 的向量维度
- `n_heads`：多头注意力的头数
- `d_ff`：前馈网络中间层维度
- `max_len`：最大序列长度
- `pad_id`：padding token 编号


In [18]:
@dataclass
class TransformerConfig:
    src_vocab_size: int = 1000
    tgt_vocab_size: int = 1000
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 512
    n_encoder_layers: int = 2
    n_decoder_layers: int = 2
    dropout: float = 0.1
    max_len: int = 256
    pad_id: int = 0


cfg = TransformerConfig()
cfg


TransformerConfig(src_vocab_size=1000, tgt_vocab_size=1000, d_model=128, n_heads=4, d_ff=512, n_encoder_layers=2, n_decoder_layers=2, dropout=0.1, max_len=256, pad_id=0)

### 小样例：配置参数怎么看

这个 cell 不只是检查参数，而是把 Transformer 最重要的尺寸关系打印出来。

你重点看：`d_model` 会被切成 `n_heads` 份，每个头的维度就是 `d_head = d_model / n_heads`。


In [ ]:
d_head = cfg.d_model // cfg.n_heads

print("模型配置：")
print(f"  d_model = {cfg.d_model}，每个 token 会变成 {cfg.d_model} 维向量")
print(f"  n_heads = {cfg.n_heads}，多头注意力会分成 {cfg.n_heads} 个头")
print(f"  d_head  = {d_head}，每个头只处理 {d_head} 维")
print(f"  d_ff    = {cfg.d_ff}，前馈网络会先扩展到 {cfg.d_ff} 维再压回 {cfg.d_model} 维")
print()
print("尺寸关系：")
print(f"  {cfg.d_model} = {cfg.n_heads} * {d_head}")
assert cfg.d_model % cfg.n_heads == 0


## 3. Mask：哪些位置可以看

这里约定：mask 中 `True` 表示可以看，`False` 表示不能看。

Transformer 常用两种 mask：

- Padding mask：不让模型关注补齐长度用的 padding。
- Causal mask：Decoder 预测当前位置时，不能偷看未来位置。


In [3]:
def make_pad_mask(tokens, pad_id=0):
    """tokens: (batch, seq_len) -> (batch, 1, 1, seq_len)"""
    return (tokens != pad_id).unsqueeze(1).unsqueeze(2)


def make_causal_mask(seq_len, device=None):
    """返回下三角 mask: (1, 1, seq_len, seq_len)"""
    mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))
    return mask.unsqueeze(0).unsqueeze(0)


x = torch.tensor([[5, 6, 0, 0], [7, 8, 9, 0]])
print(make_pad_mask(x).shape)
print(make_causal_mask(4)[0, 0].int())


torch.Size([2, 1, 1, 4])
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)


### 小样例：Mask 如何挡住 padding 和未来 token

这个例子展示两件事：

- Padding mask：`0` 是 padding，不能被注意力看到。
- Causal mask：Decoder 只能看当前位置及以前的位置，不能看未来。


In [ ]:
tokens = torch.tensor([[5, 6, 0, 0]])
pad_mask = make_pad_mask(tokens, pad_id=0)
causal_mask = make_causal_mask(4)
combined_mask = pad_mask & causal_mask

print("输入 token：")
print(tokens)
print("说明：0 是 padding")
print()
print("Padding mask，1 表示可以看，0 表示不能看：")
print(pad_mask[0, 0, 0].int())
print()
print("Causal mask，下三角表示不能偷看未来：")
print(causal_mask[0, 0].int())
print()
print("Decoder 实际使用的组合 mask：")
print(combined_mask[0, 0].int())


## 4. Embedding 和位置编码

Embedding 把 token id 映射成向量。

注意力本身不知道顺序，所以要加入位置编码。这里使用经典的正弦余弦位置编码：偶数维用 `sin`，奇数维用 `cos`。


In [4]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.scale = math.sqrt(d_model)

    def forward(self, tokens):
        return self.embedding(tokens) * self.scale


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


### 小样例：token id 如何变成向量，并加入位置信息

Embedding 只看 token 是谁；位置编码告诉模型 token 在第几个位置。

下面打印同一批 token 在加入位置编码前后的形状和前几个数值。


In [1]:
tokens = torch.tensor([[7, 8, 9]])
emb = TokenEmbedding(cfg.src_vocab_size, cfg.d_model, cfg.pad_id)
pos = PositionalEncoding(cfg.d_model, max_len=10, dropout=0.0)

x = emb(tokens)
y = pos(x)

print("输入 token id：")
print(tokens)
print()
print("Embedding 后形状：", tuple(x.shape))
print("第 1 个 token 的 embedding 前 8 维：")
print(x[0, 0, :8].detach().round(decimals=3))
print()
print("加入位置编码后形状：", tuple(y.shape))
print("第 1 个 token 加位置编码后的前 8 维：")
print(y[0, 0, :8].detach().round(decimals=3))
print()
print("位置编码本身前 3 个位置、前 8 维：")
print(pos.pe[0, :3, :8].round(decimals=3))


NameError: name 'torch' is not defined

## 5. Scaled Dot-Product Attention

注意力公式：

`Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V`

含义：`Q` 表示我想找什么，`K` 表示每个位置有什么索引，`V` 表示真正要取走的信息。


In [5]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    attn = F.softmax(scores, dim=-1)
    if dropout is not None:
        attn = dropout(attn)

    return attn @ v, attn


### 小样例：注意力权重表示“该看谁”

这里构造一个很小的注意力例子：2 个 query 去看 2 个 key/value。

你重点看 `attention weights`：每一行加起来是 1，数值越大表示越关注那个位置。


In [ ]:
q = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
k = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
v = torch.tensor([[[[10.0, 0.0], [0.0, 20.0]]]])
mask = torch.tensor([[[[True, True], [True, False]]]])

context, attn = scaled_dot_product_attention(q, k, v, mask=mask)

print("Q：")
print(q[0, 0])
print("K：")
print(k[0, 0])
print("V：")
print(v[0, 0])
print()
print("Mask，第二个 query 不能看第二个 key：")
print(mask[0, 0].int())
print()
print("Attention weights，每行表示一个 query 对所有 key 的关注比例：")
print(attn[0, 0].detach().round(decimals=3))
print("每行求和：", attn[0, 0].sum(dim=-1).detach().round(decimals=3))
print()
print("加权求和后的 context：")
print(context[0, 0].detach().round(decimals=3))


## 6. 多头注意力

多头注意力把 `d_model` 切成多个头。每个头独立做注意力，再拼回原来的维度。

形状变化：

`(batch, seq_len, d_model)` -> `(batch, heads, seq_len, d_head)` -> 注意力 -> `(batch, seq_len, d_model)`


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        b, t, _ = x.shape
        x = x.view(b, t, self.n_heads, self.d_head)
        return x.transpose(1, 2)

    def merge_heads(self, x):
        b, _, t, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(b, t, self.d_model)

    def forward(self, query, key, value, mask=None):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        context, attn = scaled_dot_product_attention(q, k, v, mask, self.dropout)
        return self.out_proj(self.merge_heads(context)), attn


### 小样例：多头注意力把同一个序列分成多个视角看

多头注意力会把 `d_model` 切成多个 head。每个 head 都会产生一张注意力矩阵。

这个例子打印输出形状和第 1 个 head 的注意力矩阵。


In [ ]:
mha = MultiHeadAttention(cfg.d_model, cfg.n_heads, dropout=0.0)
x = torch.randn(1, 4, cfg.d_model)
mask = torch.ones(1, 1, 1, 4, dtype=torch.bool)

out, attn = mha(x, x, x, mask)

print("输入 x 形状：", tuple(x.shape))
print("输出 out 形状：", tuple(out.shape))
print("注意力 attn 形状：", tuple(attn.shape))
print()
print("含义：attn.shape = (batch, heads, query_len, key_len)")
print(f"这里有 {cfg.n_heads} 个 head，每个 head 都有一张 4x4 注意力矩阵")
print()
print("第 1 个 head 的注意力矩阵：")
print(attn[0, 0].detach().round(decimals=3))


## 7. 前馈网络

注意力负责让不同位置交流信息；前馈网络负责对每个位置的向量做非线性加工。

结构是：`d_model -> d_ff -> d_model`。


In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


### 小样例：前馈网络逐位置加工向量

FeedForward 不混合不同位置的信息；它对每个位置的向量使用同一套 MLP。

下面只改动一个位置的输入，你可以看到输出也仍然按位置排列，形状不变。


In [ ]:
ffn = FeedForward(cfg.d_model, cfg.d_ff, dropout=0.0)
x = torch.zeros(1, 3, cfg.d_model)
x[0, 1, 0] = 1.0

y = ffn(x)

print("输入形状：", tuple(x.shape))
print("输出形状：", tuple(y.shape))
print()
print("输入中 3 个位置的前 6 维：")
print(x[0, :, :6])
print()
print("输出中 3 个位置的前 6 维：")
print(y[0, :, :6].detach().round(decimals=3))
print()
print("形状不变，但每个位置的向量都被 MLP 加工过。")


## 8. Encoder Layer

一个 Encoder 层包含：

1. Self-Attention：源序列内部互相看
2. Feed Forward：逐位置加工

每一块都配残差连接、LayerNorm 和 Dropout。


In [8]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        attn_out, attn = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn


### 小样例：Encoder Layer 让源序列内部互相交流

Encoder Layer 做 self-attention，所以 source 里的每个位置都可以看 source 的其他非 padding 位置。

下面打印 padding mask 和第 1 个 head 的注意力矩阵。


In [ ]:
layer = EncoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, dropout=0.0)
x = torch.randn(1, 4, cfg.d_model)
src_tokens_for_mask = torch.tensor([[11, 12, 13, 0]])
src_mask = make_pad_mask(src_tokens_for_mask, cfg.pad_id)

y, attn = layer(x, src_mask)

print("source token：", src_tokens_for_mask.tolist())
print("padding mask：", src_mask[0, 0, 0].int().tolist())
print("输入形状：", tuple(x.shape))
print("输出形状：", tuple(y.shape))
print()
print("第 1 个 head 的 self-attention：")
print(attn[0, 0].detach().round(decimals=3))
print()
print("最后一列对应 padding token，注意力应该接近 0。")


## 9. Decoder Layer

一个 Decoder 层包含：

1. Masked Self-Attention：目标序列内部看，但不能看未来
2. Cross-Attention：目标序列去看 Encoder 输出
3. Feed Forward：逐位置加工


In [10]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, memory_mask=None):
        out, self_attn = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(out))
        out, cross_attn = self.cross_attn(x, memory, memory, memory_mask)
        x = self.norm2(x + self.dropout(out))
        out = self.ffn(x)
        x = self.norm3(x + self.dropout(out))
        return x, self_attn, cross_attn


### 小样例：Decoder Layer 同时看目标历史和 Encoder 输出

Decoder Layer 有两种注意力：

- self-attention：目标序列内部看历史，不能看未来。
- cross-attention：目标序列去看 Encoder 的输出。


In [ ]:
layer = DecoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, dropout=0.0)
tgt_x = torch.randn(1, 3, cfg.d_model)
memory = torch.randn(1, 4, cfg.d_model)
tgt_mask = make_causal_mask(3)
memory_mask = torch.ones(1, 1, 1, 4, dtype=torch.bool)

y, self_attn, cross_attn = layer(tgt_x, memory, tgt_mask, memory_mask)

print("target 输入形状：", tuple(tgt_x.shape))
print("encoder memory 形状：", tuple(memory.shape))
print("decoder 输出形状：", tuple(y.shape))
print()
print("self-attention 第 1 个 head：")
print(self_attn[0, 0].detach().round(decimals=3))
print()
print("cross-attention 第 1 个 head：")
print(cross_attn[0, 0].detach().round(decimals=3))


## 10. Encoder 和 Decoder

Encoder 和 Decoder 都是把多个 layer 堆起来。

输入 token 会经过：`TokenEmbedding -> PositionalEncoding -> 多层 Transformer Layer`。


In [11]:
class Encoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = TokenEmbedding(cfg.src_vocab_size, cfg.d_model, cfg.pad_id)
        self.pos_emb = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout)
            for _ in range(cfg.n_encoder_layers)
        ])

    def forward(self, src_tokens, src_mask=None):
        x = self.pos_emb(self.token_emb(src_tokens))
        attentions = []
        for layer in self.layers:
            x, attn = layer(x, src_mask)
            attentions.append(attn)
        return x, attentions


class Decoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = TokenEmbedding(cfg.tgt_vocab_size, cfg.d_model, cfg.pad_id)
        self.pos_emb = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout)
            for _ in range(cfg.n_decoder_layers)
        ])

    def forward(self, tgt_tokens, memory, tgt_mask=None, memory_mask=None):
        x = self.pos_emb(self.token_emb(tgt_tokens))
        self_attns, cross_attns = [], []
        for layer in self.layers:
            x, self_attn, cross_attn = layer(x, memory, tgt_mask, memory_mask)
            self_attns.append(self_attn)
            cross_attns.append(cross_attn)
        return x, self_attns, cross_attns


### 小样例：Encoder 产出 memory，Decoder 读取 memory

Encoder 把 source 编码成 `memory`。

Decoder 输入 target 前缀，同时通过 cross-attention 读取 `memory`。


In [ ]:
encoder = Encoder(cfg)
decoder = Decoder(cfg)
src_tokens = torch.tensor([[4, 5, 6, 0]])
tgt_tokens = torch.tensor([[1, 8, 9]])

src_mask = make_pad_mask(src_tokens, cfg.pad_id)
tgt_mask = make_pad_mask(tgt_tokens, cfg.pad_id) & make_causal_mask(tgt_tokens.size(1))

memory, enc_attns = encoder(src_tokens, src_mask)
dec_out, self_attns, cross_attns = decoder(tgt_tokens, memory, tgt_mask, src_mask)

print("source token：", src_tokens.tolist())
print("target token：", tgt_tokens.tolist())
print("Encoder 输出 memory 形状：", tuple(memory.shape))
print("Decoder 输出形状：", tuple(dec_out.shape))
print()
print("Encoder 层数：", len(enc_attns))
print("Decoder self-attention 层数：", len(self_attns))
print("Decoder cross-attention 层数：", len(cross_attns))
print()
print("最后一层 cross-attention 第 1 个 head：")
print(cross_attns[-1][0, 0].detach().round(decimals=3))


## 11. 完整 Transformer

完整流程：

1. 给 source 和 target 创建 mask
2. Encoder 编码 source
3. Decoder 一边看已经生成的 target，一边通过 cross-attention 看 source
4. 最后一层线性层输出词表 logits

输出形状：`(batch, tgt_len, tgt_vocab_size)`。


In [12]:
class Transformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg)
        self.generator = nn.Linear(cfg.d_model, cfg.tgt_vocab_size)

    def make_masks(self, src_tokens, tgt_tokens):
        src_mask = make_pad_mask(src_tokens, self.cfg.pad_id)
        tgt_pad_mask = make_pad_mask(tgt_tokens, self.cfg.pad_id)
        tgt_causal_mask = make_causal_mask(tgt_tokens.size(1), tgt_tokens.device)
        tgt_mask = tgt_pad_mask & tgt_causal_mask
        memory_mask = src_mask
        return src_mask, tgt_mask, memory_mask

    def forward(self, src_tokens, tgt_tokens):
        src_mask, tgt_mask, memory_mask = self.make_masks(src_tokens, tgt_tokens)
        memory, enc_attn = self.encoder(src_tokens, src_mask)
        dec_out, dec_self_attn, dec_cross_attn = self.decoder(tgt_tokens, memory, tgt_mask, memory_mask)
        logits = self.generator(dec_out)
        return {
            "logits": logits,
            "encoder_attn": enc_attn,
            "decoder_self_attn": dec_self_attn,
            "decoder_cross_attn": dec_cross_attn,
        }


### 小样例：完整 Transformer 输出每个位置的词表分数

完整模型输出 `logits`，形状是 `(batch, tgt_len, tgt_vocab_size)`。

对最后一维取 `argmax`，就能得到每个位置当前预测的 token id。


In [ ]:
test_model = Transformer(cfg)
src_tokens = torch.tensor([[4, 5, 6, 0]])
tgt_tokens = torch.tensor([[1, 8, 9]])

out = test_model(src_tokens, tgt_tokens)
logits = out["logits"]
pred_ids = logits.argmax(dim=-1)

print("source token：", src_tokens.tolist())
print("target 输入 token：", tgt_tokens.tolist())
print("logits 形状：", tuple(logits.shape))
print("预测 token id 形状：", tuple(pred_ids.shape))
print("当前随机初始化模型的预测 token id：")
print(pred_ids.tolist())
print()
print("注意：模型还没训练，所以预测值现在没有语义，只用来看完整流程。")


## 12. 前向传播检查

随机造一批 token，确认模型能跑通，并检查关键张量形状。


In [13]:
model = Transformer(cfg).to(DEVICE)

batch_size, src_len, tgt_len = 2, 7, 6
src_tokens = torch.randint(1, cfg.src_vocab_size, (batch_size, src_len), device=DEVICE)
tgt_input = torch.randint(1, cfg.tgt_vocab_size, (batch_size, tgt_len), device=DEVICE)

src_tokens[0, -2:] = cfg.pad_id
tgt_input[1, -1:] = cfg.pad_id

out = model(src_tokens, tgt_input)
logits = out["logits"]

print("src_tokens:", src_tokens.shape)
print("tgt_input:", tgt_input.shape)
print("logits:", logits.shape)
print("encoder attention:", out["encoder_attn"][0].shape)
print("decoder self attention:", out["decoder_self_attn"][0].shape)
print("decoder cross attention:", out["decoder_cross_attn"][0].shape)


src_tokens: torch.Size([2, 7])
tgt_input: torch.Size([2, 6])
logits: torch.Size([2, 6, 1000])
encoder attention: torch.Size([2, 4, 7, 7])
decoder self attention: torch.Size([2, 4, 6, 6])
decoder cross attention: torch.Size([2, 4, 6, 7])


## 13. 训练损失

训练时通常使用 teacher forcing：

- Decoder 输入：目标序列右移一位
- 预测目标：原始目标序列

`ignore_index=pad_id` 表示 padding 位置不参与 loss。


In [14]:
tgt_output = torch.randint(1, cfg.tgt_vocab_size, (batch_size, tgt_len), device=DEVICE)
tgt_output[1, -1:] = cfg.pad_id

loss = F.cross_entropy(
    logits.reshape(-1, cfg.tgt_vocab_size),
    tgt_output.reshape(-1),
    ignore_index=cfg.pad_id,
)
assert torch.isfinite(loss)
print("loss:", float(loss.detach()))
print("loss test passed")


loss: 7.139402866363525


C:\Users\91638\AppData\Local\Temp\ipykernel_31084\3206114288.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  print("loss:", float(loss))


## 14. 小玩具训练：复制序列

这个 toy task 让模型把输入序列复制到输出。它不追求效果，只用来展示完整训练循环。

步骤：生成随机 source，把 target 设成 source，构造右移后的 Decoder 输入，计算 loss，反向传播更新参数。


In [15]:
BOS_ID = 1
VOCAB_SIZE = 50
PAD_ID = 0

small_cfg = TransformerConfig(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=64,
    n_heads=4,
    d_ff=128,
    n_encoder_layers=2,
    n_decoder_layers=2,
    dropout=0.1,
    max_len=32,
    pad_id=PAD_ID,
)

small_model = Transformer(small_cfg).to(DEVICE)
optimizer = torch.optim.AdamW(small_model.parameters(), lr=3e-4)


def make_copy_batch(batch_size=32, seq_len=10):
    src = torch.randint(2, VOCAB_SIZE, (batch_size, seq_len), device=DEVICE)
    tgt_y = src.clone()
    bos = torch.full((batch_size, 1), BOS_ID, dtype=torch.long, device=DEVICE)
    tgt_x = torch.cat([bos, tgt_y[:, :-1]], dim=1)
    return src, tgt_x, tgt_y


small_model.train()
for step in range(1, 21):
    src, tgt_x, tgt_y = make_copy_batch()
    logits = small_model(src, tgt_x)["logits"]
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt_y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(small_model.parameters(), max_norm=1.0)
    optimizer.step()

    if step == 1 or step % 5 == 0:
        print(f"step {step:02d} | loss {loss.item():.4f}")


step 01 | loss 4.1027
step 05 | loss 4.0294
step 10 | loss 4.0510
step 15 | loss 3.9503
step 20 | loss 3.9605


## 15. 贪心解码

推理时没有真实答案，所以 Decoder 要一步一步生成。

每一步把目前已经生成的 token 输入模型，然后取最后一个位置概率最大的 token 作为下一步输出。


In [16]:
@torch.no_grad()
def greedy_decode(model, src_tokens, bos_id=1, max_len=10):
    model.eval()
    b = src_tokens.size(0)
    generated = torch.full((b, 1), bos_id, dtype=torch.long, device=src_tokens.device)

    for _ in range(max_len):
        logits = model(src_tokens, generated)["logits"]
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

    return generated[:, 1:]


src, _, tgt_y = make_copy_batch(batch_size=3, seq_len=10)
pred = greedy_decode(small_model, src, bos_id=BOS_ID, max_len=10)

assert pred.shape == tgt_y.shape
print("source:")
print(src.cpu())
print("target:")
print(tgt_y.cpu())
print("predict:")
print(pred.cpu())
print("greedy decode test passed")


SyntaxError: unterminated string literal (detected at line 18) (1925172894.py, line 18)

## 16. 总结

Transformer 的主干可以记成这几件事：

- Embedding：token id 变向量
- Positional Encoding：加入顺序信息
- Self-Attention：同一序列内部交换信息
- Cross-Attention：Decoder 查询 Encoder 输出
- Feed Forward：逐位置加工特征
- Residual + LayerNorm：稳定深层训练
- Mask：控制哪些位置可以被看见

读代码时一直盯住张量形状，Transformer 会清楚很多。
